# 03 — Image Preprocessing

## Objectives
- Validate the COCO image files and dataset manifests.
- Inspect image dimensions, formats, and colour modes.
- Identify unreadable or invalid images.
- Define reproducible image preprocessing operations.
- Save preprocessing reports and optional processed outputs separately.
- Preserve the original COCO dataset without modification.

## Dataset
MS COCO 2017 — official training and validation splits.

## Reproducibility
Random seed: 42
Original images are read-only inputs.

Setup

In [1]:
import antigravity

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Tests and check ups

In [3]:
from pathlib import Path

# Project workspace
WORKSPACE = Path(
    "/content/drive/MyDrive/image-captioning-rag"
)

# Original COCO dataset — do not modify these files
RAW_ROOT = WORKSPACE / "00_original_data"

TRAIN_IMAGES = RAW_ROOT / "train2017"
VAL_IMAGES = RAW_ROOT / "val2017"
ANNOTATIONS = RAW_ROOT / "annotations"

# Caption annotations
CAPTIONS_TRAIN = ANNOTATIONS / "captions_train2017.json"
CAPTIONS_VAL = ANNOTATIONS / "captions_val2017.json"

# Instance annotations
INSTANCES_TRAIN = ANNOTATIONS / "instances_train2017.json"
INSTANCES_VAL = ANNOTATIONS / "instances_val2017.json"

# Outputs for this notebook
OUTPUT_DIR = WORKSPACE / "reports" / "preprocessing"
PLOTS_DIR = OUTPUT_DIR / "plots"
QC_DIR = OUTPUT_DIR / "quality_control"

for directory in [OUTPUT_DIR, PLOTS_DIR, QC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Workspace:", WORKSPACE)
print("Original dataset:", RAW_ROOT)
print("Preprocessing outputs:", OUTPUT_DIR)

Workspace: /content/drive/MyDrive/image-captioning-rag
Original dataset: /content/drive/MyDrive/image-captioning-rag/00_original_data
Preprocessing outputs: /content/drive/MyDrive/image-captioning-rag/reports/preprocessing


In [4]:
import json
import random
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile, UnidentifiedImageError
from collections import Counter
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Do not accept truncated images as valid by default
ImageFile.LOAD_TRUNCATED_IMAGES = False

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

logger = logging.getLogger("image_preprocessing")

print("Libraries imported.")
print("Random seed:", SEED)
print("Pillow version:", __import__("PIL").__version__)

Libraries imported.
Random seed: 42
Pillow version: 11.3.0


In [5]:
required_paths = {
    "Training images": TRAIN_IMAGES,
    "Validation images": VAL_IMAGES,
    "Annotations": ANNOTATIONS,
    "Training captions": CAPTIONS_TRAIN,
    "Validation captions": CAPTIONS_VAL,
    "Training instances": INSTANCES_TRAIN,
    "Validation instances": INSTANCES_VAL,
}

for name, path in required_paths.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:7} | {name}: {path}")

missing = [
    name for name, path in required_paths.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required paths: " + ", ".join(missing)
    )

print("\nAll required inputs are available.")

OK      | Training images: /content/drive/MyDrive/image-captioning-rag/00_original_data/train2017
OK      | Validation images: /content/drive/MyDrive/image-captioning-rag/00_original_data/val2017
OK      | Annotations: /content/drive/MyDrive/image-captioning-rag/00_original_data/annotations
OK      | Training captions: /content/drive/MyDrive/image-captioning-rag/00_original_data/annotations/captions_train2017.json
OK      | Validation captions: /content/drive/MyDrive/image-captioning-rag/00_original_data/annotations/captions_val2017.json
OK      | Training instances: /content/drive/MyDrive/image-captioning-rag/00_original_data/annotations/instances_train2017.json
OK      | Validation instances: /content/drive/MyDrive/image-captioning-rag/00_original_data/annotations/instances_val2017.json

All required inputs are available.


In [7]:
SPLIT_DIR = WORKSPACE / "reports" / "split_statistics"

TRAIN_MANIFEST = SPLIT_DIR / "train_manifest.csv"
VAL_MANIFEST = SPLIT_DIR / "val_manifest.csv"

if TRAIN_MANIFEST.exists() and VAL_MANIFEST.exists():
    train_df = pd.read_csv(TRAIN_MANIFEST)
    val_df = pd.read_csv(VAL_MANIFEST)

    print("Loaded existing split manifests.")

else:
    # Fallback: build manifests from COCO caption metadata.
    # This does not move, copy, or split the image files.
    with open(CAPTIONS_TRAIN, "r", encoding="utf-8") as f:
        train_json = json.load(f)

    with open(CAPTIONS_VAL, "r", encoding="utf-8") as f:
        val_json = json.load(f)

    train_df = pd.DataFrame([
        {
            "image_id": image["id"],
            "filename": image["file_name"],
            "split": "train"
        }
        for image in train_json["images"]
    ])

    val_df = pd.DataFrame([
        {
            "image_id": image["id"],
            "filename": image["file_name"],
            "split": "val"
        }
        for image in val_json["images"]
    ])

print("Training manifest rows:", len(train_df))
print("Validation manifest rows:", len(val_df))

display(train_df.head())
display(val_df.head())

Training manifest rows: 118287
Validation manifest rows: 5000


,image_id,filename,split
0,391895,000000391895.jpg,train
1,522418,000000522418.jpg,train
2,184613,000000184613.jpg,train
3,318219,000000318219.jpg,train
4,554625,000000554625.jpg,train


,image_id,filename,split
0,397133,000000397133.jpg,val
1,37777,000000037777.jpg,val
2,252219,000000252219.jpg,val
3,87038,000000087038.jpg,val
4,174482,000000174482.jpg,val


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [8]:
def validate_manifest(manifest, image_dir, split_name):
    required_columns = {"image_id", "filename"}

    if not required_columns.issubset(manifest.columns):
        raise ValueError(
            f"{split_name} manifest must contain "
            f"{sorted(required_columns)}"
        )

    if manifest["image_id"].duplicated().any():
        raise ValueError(
            f"Duplicate image IDs in {split_name} manifest."
        )

    if manifest["filename"].duplicated().any():
        raise ValueError(
            f"Duplicate filenames in {split_name} manifest."
        )

    exists = manifest["filename"].map(
        lambda filename: (image_dir / str(filename)).is_file()
    )

    missing_files = manifest.loc[~exists, "filename"].tolist()

    print(f"\n{split_name}:")
    print("Manifest records:", len(manifest))
    print("Existing files:", int(exists.sum()))
    print("Missing files:", len(missing_files))

    if missing_files:
        print("Examples:", missing_files[:10])

    return missing_files


missing_train = validate_manifest(
    train_df, TRAIN_IMAGES, "Training"
)

missing_val = validate_manifest(
    val_df, VAL_IMAGES, "Validation"
)

if missing_train or missing_val:
    logger.warning(
        "Some manifest entries do not match available image files."
    )
else:
    print("\nAll manifest filenames were found.")


Training:
Manifest records: 118287
Existing files: 61705
Missing files: 56582
Examples: ['000000391895.jpg', '000000184613.jpg', '000000005802.jpg', '000000222564.jpg', '000000118113.jpg', '000000193271.jpg', '000000224736.jpg', '000000403013.jpg', '000000374628.jpg', '000000384213.jpg']



Validation:
Manifest records: 5000
Existing files: 5000
Missing files: 0


Long one to run

In [10]:
from pathlib import Path

print("Training folder:", TRAIN_IMAGES)
print("Folder exists:", TRAIN_IMAGES.is_dir())

# Count image files directly inside train2017
direct_images = list(TRAIN_IMAGES.glob("*.jpg"))

print("JPEG images directly inside train2017:", len(direct_images))

# Check whether images were accidentally extracted into a nested folder
nested_images = list(TRAIN_IMAGES.rglob("*.jpg"))

print("JPEG images including subfolders:", len(nested_images))

# Display a few entries from the folder
print("\nFirst entries in training folder:")
for item in list(TRAIN_IMAGES.iterdir())[:15]:
    print(item.name, "| Directory:", item.is_dir())

Training folder: /content/drive/MyDrive/image-captioning-rag/00_original_data/train2017
Folder exists: True
JPEG images directly inside train2017: 61705
JPEG images including subfolders: 61705

First entries in training folder:
000000342159.jpg | Directory: False
000000549996.jpg | Directory: False
000000145767.jpg | Directory: False
000000383764.jpg | Directory: False
000000169154.jpg | Directory: False
000000170990.jpg | Directory: False
000000220430.jpg | Directory: False
000000078640.jpg | Directory: False
000000401622.jpg | Directory: False
000000063476.jpg | Directory: False
000000436693.jpg | Directory: False
000000232280.jpg | Directory: False
000000514404.jpg | Directory: False
000000393721.jpg | Directory: False
000000222056.jpg | Directory: False


Start preprocessing

In [11]:
from PIL import Image, ImageOps

def load_rgb_image(path):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        return image.convert("RGB")

Purpose: Core Image Loading Helper.

Operations: Defines load_rgb_image(path). It uses Pillow's ImageOps.exif_transpose to respect capture orientation metadata and standardizes images into the RGB color space.

In [13]:
# Define output root directory and build image indexes
PROCESSED_ROOT = WORKSPACE / "reports" / "preprocessing"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

# Create a mapping from image ID to image records from COCO json
train_records_by_id = {
    image["id"]: image
    for image in train_json["images"]
}

# Filter to find image IDs that are actually present in our subset folder
existing_train_filenames = set(item.name for item in TRAIN_IMAGES.iterdir() if item.is_file())

# Filter our training IDs to include only those whose images actually exist in train2017
available_train_ids = [
    image_id for image_id, record in train_records_by_id.items()
    if record["file_name"] in existing_train_filenames
]

# Define the knowledge base IDs (using a subset of 1000 existing training images for demonstration)
random.seed(SEED)
knowledge_base_ids = random.sample(
    available_train_ids,
    min(1000, len(available_train_ids))
)

print(f"Indexed {len(train_records_by_id)} training images.")
print(f"Found {len(available_train_ids)} available images locally.")
print(f"Selected {len(knowledge_base_ids)} knowledge base image IDs.")

Indexed 118287 training images.
Found 61705 available images locally.
Selected 1000 knowledge base image IDs.


Purpose: Training Index Creation and KB Sampling.

Operations: Maps COCO IDs to JSON images and intersects them with actual locally available files to sample 1,000 unique images for the RAG Knowledge Base.

Results: Successfully indexed 118,287 metadata records, cross-referenced them with 61,705 local files, and sampled 1,000 targets.

In [14]:
image_manifest = pd.DataFrame([
    {
        "image_id": image_id,
        "file_name": train_records_by_id[image_id]["file_name"],
        "image_path": str(
            TRAIN_IMAGES
            / train_records_by_id[image_id]["file_name"]
        ),
        "split": "knowledge_base",
    }
    for image_id in knowledge_base_ids
])

image_manifest.to_parquet(
    PROCESSED_ROOT / "knowledge_base_images.parquet",
    index=False,
)

display(image_manifest.head())

,image_id,file_name,image_path,split
0,149996,000000149996.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base
1,240279,000000240279.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base
2,17364,000000017364.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base
3,277122,000000277122.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base
4,196244,000000196244.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base


Purpose: Metadata Parquet Serialization (Initial).

Operations: Generates a DataFrame containing image paths for the 1,000 selected Knowledge Base images and saves it to a Parquet file.

In [15]:
import random
import pandas as pd

# Output directory
PROCESSED_ROOT = WORKSPACE / "reports" / "preprocessing"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

# Map COCO image IDs to metadata
train_records_by_id = {
    image["id"]: image
    for image in train_json["images"]
}

# Find locally available image filenames
existing_train_filenames = {
    item.name
    for item in TRAIN_IMAGES.iterdir()
    if item.is_file()
}

# Keep only image IDs whose files exist locally
available_train_ids = [
    image_id
    for image_id, record in train_records_by_id.items()
    if record["file_name"] in existing_train_filenames
]

# Reproducible selection
random.seed(SEED)
KB_SIZE = min(1000, len(available_train_ids))

knowledge_base_ids = random.sample(
    available_train_ids,
    KB_SIZE
)

# Create manifest
image_manifest = pd.DataFrame([
    {
        "image_id": image_id,
        "file_name": train_records_by_id[image_id]["file_name"],
        "image_path": str(
            TRAIN_IMAGES /
            train_records_by_id[image_id]["file_name"]
        ),
        "split": "knowledge_base",
    }
    for image_id in knowledge_base_ids
])

# Validate
assert image_manifest["image_id"].is_unique
assert image_manifest["image_path"].map(
    lambda path: Path(path).is_file()
).all()

# Save
manifest_path = PROCESSED_ROOT / "knowledge_base_images.parquet"
image_manifest.to_parquet(manifest_path, index=False)

print("Training annotation records:", len(train_records_by_id))
print("Available local training images:", len(available_train_ids))
print("Knowledge base images selected:", len(image_manifest))
print("Manifest saved to:", manifest_path)

display(image_manifest.head())

Training annotation records: 118287
Available local training images: 61705
Knowledge base images selected: 1000
Manifest saved to: /content/drive/MyDrive/image-captioning-rag/reports/preprocessing/knowledge_base_images.parquet


,image_id,file_name,image_path,split
0,149996,000000149996.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base
1,240279,000000240279.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base
2,17364,000000017364.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base
3,277122,000000277122.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base
4,196244,000000196244.jpg,/content/drive/MyDrive/image-captioning-rag/00...,knowledge_base


Purpose: Robust Split Manifest Generation & Integrity Check.

Operations: Refined version of Cell 14/15 that builds the Knowledge Base split manifest (knowledge_base_images.parquet) and applies rigorous assertions to guarantee zero missing files.

Results: Validated that all 1,000 files are fully intact locally and exported the manifest to your Drive reports folder.

In [17]:
with open(INSTANCES_TRAIN, "r", encoding="utf-8") as f:
    instances_train_json = json.load(f)

# Map category ID to category name
category_names = {
    cat["id"]: cat["name"]
    for cat in instances_train_json["categories"]
}

# Set of knowledge base image IDs
kb_ids_set = set(knowledge_base_ids)

# Filter annotations belonging to our selected knowledge base images
kb_objects = [
    ann for ann in instances_train_json["annotations"]
    if ann["image_id"] in kb_ids_set
]

print(f"Loaded {len(category_names)} categories.")
print(f"Extracted {len(kb_objects)} object annotations for our knowledge base images.")

Loaded 80 categories.
Extracted 7284 object annotations for our knowledge base images.


Purpose: COCO Object Annotations Filter.

Operations: Loads object instance annotations and maps category IDs (e.g., 18) to human-readable names (dog). It filters annotations to isolate those corresponding to the 1,000 selected Knowledge Base images.

Results: Loaded 80 default COCO categories and filtered 7,284 object instances from the dataset.

In [18]:
metadata_rows = []

for ann in kb_objects:
    metadata_rows.append({
        "image_id": ann["image_id"],
        "object_annotation_id": ann["id"],
        "category_id": ann["category_id"],
        "category": category_names[ann["category_id"]],
        "bbox_x": ann["bbox"][0],
        "bbox_y": ann["bbox"][1],
        "bbox_width": ann["bbox"][2],
        "bbox_height": ann["bbox"][3],
        "area": ann["area"],
        "iscrowd": ann["iscrowd"],
        "source": "COCO instances_train2017",
    })

object_metadata = pd.DataFrame(metadata_rows)

object_metadata.to_parquet(
    PROCESSED_ROOT / "knowledge_base_objects.parquet",
    index=False,
)

print("Object metadata rows:", len(object_metadata))

Object metadata rows: 7284


Purpose: Ground-truth Object Parquet Export.

Operations: Extracts structured bounding box coordinates (bbox_x, bbox_y, bbox_width, bbox_height), object areas, and crowd parameters, saving them into knowledge_base_objects.parquet.

Results: Successfully exported 7,284 object rows for seamless spatial indexing.

In [20]:
def validate_image_paths(image_dir, records):
    missing_ids = []
    corrupt_ids = []
    for r in records:
        path = image_dir / r["file_name"]
        if not path.is_file():
            missing_ids.append(r["id"])
        else:
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception:
                corrupt_ids.append(r["id"])
    return {"missing_ids": missing_ids, "corrupt_ids": corrupt_ids}

# Map COCO validation image IDs to metadata
val_records_by_id = {
    image["id"]: image
    for image in val_json["images"]
}

# Filter to find validation image IDs that are actually present in our val2017 folder
existing_val_filenames = set(item.name for item in VAL_IMAGES.iterdir() if item.is_file())
available_val_ids = [
    image_id for image_id, record in val_records_by_id.items()
    if record["file_name"] in existing_val_filenames
]

# Sample validation and test splits from the available validation set
random.seed(SEED)
validation_ids = random.sample(available_val_ids, min(500, len(available_val_ids)))

# Ensure test IDs do not overlap with validation IDs
remaining_val_ids = list(set(available_val_ids) - set(validation_ids))
test_ids = random.sample(remaining_val_ids, min(500, len(remaining_val_ids)))

print(f"Set up validation split with {len(validation_ids)} images.")
print(f"Set up test split with {len(test_ids)} images.")

Set up validation split with 500 images.
Set up test split with 500 images.


Purpose: Path Integrity Helper & Validation / Test Splitting.

Operations:
Establishes validate_image_paths() which uses PIL.Image.verify() to audit image structure for underlying file corruption.

Indexes the local validation directory (val2017) and splits it to sample 500 images for a Model Validation split and 500 images for an Evaluation Test split (ensuring zero overlap).

In [22]:
kb_records = [
    train_records_by_id[i]
    for i in knowledge_base_ids
]

val_records = [
    val_records_by_id[i]
    for i in validation_ids
]

test_records = [
    val_records_by_id[i]
    for i in test_ids
]

kb_audit = validate_image_paths(TRAIN_IMAGES, kb_records)
val_audit = validate_image_paths(VAL_IMAGES, val_records)
test_audit = validate_image_paths(VAL_IMAGES, test_records)

for name, result in [
    ("knowledge_base", kb_audit),
    ("validation", val_audit),
    ("test", test_audit),
]:
    print(
        name,
        "missing:", len(result["missing_ids"]),
        "corrupt:", len(result["corrupt_ids"]),
    )

knowledge_base missing: 0 corrupt: 0
validation missing: 0 corrupt: 0
test missing: 0 corrupt: 0


Purpose: Comprehensive Dataset Audit & Debugging.

Operations: Compiles exact metadata records for the Knowledge Base, Validation, and Test splits, running them through the PIL image verification helper to test for missing or corrupt files.

Results: 0 missing, 0 corrupt files detected across all datasets. All subsets are clean and structurally uncorrupted!